# Practical Problems: FIND-S and Candidate-Elimination Algorithms

This notebook demonstrates the implementation of:
1. **FIND-S Algorithm** to find the most specific hypothesis.
2. **Candidate-Elimination Algorithm** to output the version space (most specific and most general hypotheses) consistent with the training examples.

In [5]:
import pandas as pd
import numpy as np

csv_path = '/content/enjoysport - enjoysport.csv'

print(f"Loading dataset from: {csv_path}")
df = pd.read_csv(csv_path)
display(df)

Loading dataset from: /content/enjoysport - enjoysport.csv


,sky,airtemp,humidity,wind,water,forcast,enjoysport
0,sunny,warm,normal,strong,warm,same,yes
1,sunny,warm,high,strong,warm,same,yes
2,rainy,cold,high,strong,warm,change,no
3,sunny,warm,high,strong,cool,change,yes


## Practical Problem 1: FIND-S Algorithm

Find-S finds the most specific hypothesis that fits all the positive training examples.

In [6]:
def find_s(concepts, target):
    # Initialize the most specific hypothesis with the first positive example
    for i, val in enumerate(target):
        if val.lower() in ['yes', '1', 'true']:
            specific_h = concepts[i].copy()
            break

    print("Initialization of specific_h:", specific_h)

    for i, val in enumerate(concepts):
        if target[i].lower() in ['yes', '1', 'true']:
            for x in range(len(specific_h)):
                if val[x] != specific_h[x]:
                    specific_h[x] = '?'
            print(f"After sample {i+1}: {specific_h}")

    return specific_h

# Extract concepts and target
concepts = np.array(df.iloc[:, :-1])
target = np.array(df.iloc[:, -1])

print("Applying FIND-S Algorithm:")
final_hypothesis = find_s(concepts, target)
print("\nFinal Most Specific Hypothesis:", final_hypothesis)

Applying FIND-S Algorithm:
Initialization of specific_h: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
After sample 1: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
After sample 2: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
After sample 4: ['sunny' 'warm' '?' 'strong' '?' '?']

Final Most Specific Hypothesis: ['sunny' 'warm' '?' 'strong' '?' '?']


## Practical Problem 2: Candidate-Elimination Algorithm

Candidate-Elimination computes the version space containing all hypotheses consistent with the training examples.

In [7]:
def candidate_elimination(concepts, target):
    num_attributes = len(concepts[0])

    # Initialize Specific and General Boundaries
    specific_h = ["0"] * num_attributes
    general_h = [["?"] * num_attributes for _ in range(num_attributes)]

    # Find first positive example to initialize specific hypothesis
    for i, val in enumerate(target):
        if val.lower() in ['yes', '1', 'true']:
            specific_h = concepts[i].copy()
            break

    general_h = [["?"] * num_attributes for _ in range(num_attributes)]

    print(f"Initialization:")
    print(f"S0: {specific_h}")
    print(f"G0: {general_h}\n")

    for i, h in enumerate(concepts):
        if target[i].lower() in ['yes', '1', 'true']:
            # Update Specific Boundary for Positive Example
            for x in range(num_attributes):
                if h[x] != specific_h[x]:
                    specific_h[x] = '?'
                    # Filter general boundary consistent with new specific boundary
                    for g in general_h:
                        if g[x] != '?' and g[x] != specific_h[x]:
                            general_h.remove(g)
        else:
            # Update General Boundary for Negative Example
            for x in range(num_attributes):
                if h[x] != specific_h[x]:
                    # Create a specialized hypothesis
                    g_temp = ['?'] * num_attributes
                    g_temp[x] = specific_h[x]
                    if g_temp not in general_h:
                        general_h.append(g_temp)
                else:
                    # Remove incompatible general hypotheses
                    for g in list(general_h):
                        if g[x] == h[x]:
                            general_h.remove(g)

        # Remove any purely generic default placeholders or redundant entries
        general_h = [g for g in general_h if g != ['?'] * num_attributes]

        print(f"Iteration {i+1} (Sample {i+1} is {target[i]}):")
        print(f"S{i+1}: {specific_h}")
        print(f"G{i+1}: {general_h}\n")

    return specific_h, general_h

s_final, g_final = candidate_elimination(concepts, target)
print("Final Specific Boundary (S):", s_final)
print("Final General Boundary (G):", g_final)

Initialization:
S0: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
G0: [['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', '?']]

Iteration 1 (Sample 1 is yes):
S1: ['sunny' 'warm' 'normal' 'strong' 'warm' 'same']
G1: []

Iteration 2 (Sample 2 is yes):
S2: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
G2: []

Iteration 3 (Sample 3 is no):
S3: ['sunny' 'warm' '?' 'strong' 'warm' 'same']
G3: [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'same']]

Iteration 4 (Sample 4 is yes):
S4: ['sunny' 'warm' '?' 'strong' '?' '?']
G4: [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?']]

Final Specific Boundary (S): ['sunny' 'warm' '?' 'strong' '?' '?']
Final General Boundary (G): [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?']]
